# HFSS Validation around `x_optima`

`lib_gp.robust_lcb_on_J` で計算している

```python
posterior_mu_sample_std = sqrt(sum((mu_Z - mu_J) ** 2) / (M - 1))
```

は GP サロゲートの posterior mean `mu_Z` 上の標本標準偏差です。この notebook では同じ標本標準偏差の計算を、実際の HFSS 評価値 `S11` に置き換えて実施します。

## 基本設計

1. `main.ipynb` と同じ config / `Backbone` / HFSS 連携を初期化する。
2. 入力ベクトル `x_optima` の周りに Gaussian 摂動を 100 点生成する。
   - デフォルトは robust LCB と同じ考え方で、各次元の標準偏差を `PERTURBATION_STD_RATIO * (upper_bounds - lower_bounds)` とする。
   - `Sigma` を直接指定したい場合は `CUSTOM_SIGMA` を設定する。
   - bounds 外に出た点は `lib_gp.sample_input_perturbations` と同じく clip する。
3. 各摂動点 `Z_i` を HFSS で評価し、`S11_i` を保存する。
4. `mu_sample = mean(S11_i)` と `mu_sample_std = sqrt(sum((S11_i - mu_sample)**2)/(M-1))` を出力する。
5. run directory に `validation_samples.csv` と `validation_summary.csv` を保存する。

## 実行前に確定が必要な情報

- `x_optima` は空配列 `[]` のままでは実行できません。`cfg.n_params` と同じ長さの full parameter vector を設定してください。
- Gaussian 摂動の分散が未指定です。この notebook では暫定値として `PERTURBATION_STD_RATIO = 0.01` を使います。実験条件として別の標準偏差または共分散 `Sigma` が必要なら変更してください。
- HFSS 側の監視プロセスが `_config_HFSS.json` / generated STEP files / `temp_hfss_export.csv` の既存フローに対応して起動している必要があります。


In [ ]:
from pathlib import Path
import json
import os
import time

import numpy as np
import pandas as pd

import lib_config as config
import lib_backbone
import lib_gp
import lib_RFdesign

%load_ext autoreload
%autoreload 2

In [ ]:
# main.ipynb と同じ初期化
_config = config._loadConfig(Path("./_config.toml"))
app_config = config.initParams(_config, debug=True)

backbone = lib_backbone.Backbone(config=app_config)
base_dir = app_config.env.dir_base
backbone.initStorer()

cfg = app_config.hfss
ROUND_DECIMALS = app_config.runtime.round_decimals

LOWER_BOUNDS = np.asarray(cfg.lower_bounds, dtype=float)
UPPER_BOUNDS = np.asarray(cfg.upper_bounds, dtype=float)
PARAM_NAMES = list(cfg.param_names)
BOUNDS = np.vstack([LOWER_BOUNDS, UPPER_BOUNDS]).T

model_paths, model_paths_str = backbone._get_path_models()
RESULTS_FILE = str(backbone._get_dir_run() / Path(_config["io"]["filename_output"]))
TEMP_FILE = str(backbone._get_dir_run() / Path(_config["io"]["filename_temp"]))

print("Run directory:", backbone._get_dir_run())
print("n_params:", cfg.n_params)
print("PARAM_NAMES:", PARAM_NAMES)

In [ ]:
def round_params(params, decimals=ROUND_DECIMALS):
    return np.round(np.asarray(params, dtype=float).flatten(), decimals)


def getResult(input_params, param_names, temp_hfss_path, result_file_path):
    """Read one HFSS temp result and append it to the run-level result CSV."""
    df_temp = pd.read_csv(temp_hfss_path)
    header_flag = not os.path.exists(result_file_path)

    try:
        s11_value = df_temp.iloc[-1, -1]
        result_row = dict(zip(param_names, input_params))
        result_row["S11"] = s11_value

        df_result = pd.DataFrame([result_row])
        df_result.to_csv(result_file_path, mode="a", header=header_flag, index=False)

        try:
            os.remove(temp_hfss_path)
        except OSError:
            pass
        return True

    except Exception as e:
        print(f"[Error][getResult] Failed to process result: {e}")
        return False


def target_hfss(_config_temp, sim_id, param_names, params):
    """Evaluate one rounded parameter vector through the existing HFSS workflow."""
    params = round_params(params)
    backbone.call_subroutine(
        _config_temp,
        sim_id,
        param_names,
        params,
        value_fmt=f"{{:.{ROUND_DECIMALS}f}}",
    )
    getResult(params, param_names, TEMP_FILE, RESULTS_FILE)
    df_result = pd.read_csv(RESULTS_FILE)
    return float(df_result.iloc[-1]["S11"])


def evaluate_hfss(param_names, params):
    """Wrapper returning both scalar S11 and a row dictionary for validation logs."""
    params = round_params(params)
    sim_id = backbone._getSimulationID()
    y = target_hfss(_config_temp, sim_id, param_names, params)
    row = dict(zip(param_names, params))
    row["S11"] = float(np.round(y, ROUND_DECIMALS))
    row["sim_id"] = sim_id
    return y, row

In [ ]:
# HFSS watcher 用の run-specific config を main.ipynb と同じ形式で作成する
_config_temp = {
    "n_simulation": 100,
    "n_repeats": 1,
    "WATCH_DIR": str(backbone._get_dir_run()),
    "INPUT_FILE": str(backbone._get_dir_run() / Path(_config["io"]["filename_input"])),
    "MODEL_FILE": model_paths_str,
    "RESULTS_FILE": RESULTS_FILE,
    "TEMP_FILE": TEMP_FILE,
    "DONE_FLAG_FILE": str(backbone._get_dir_run() / Path("hfss.done")),
}

done_flag_path = Path(_config_temp["DONE_FLAG_FILE"])
done_flag_path.unlink(missing_ok=True)

with open(base_dir / Path("_config_HFSS.json"), "w") as f:
    json.dump(_config_temp, f, indent=4)

print(f"Temporarily updated '{base_dir / Path('_config_HFSS.json')}' with run-specific WATCH_DIR for HFSS.")

In [ ]:
# ===== User settings =====
# TODO: cfg.n_params と同じ長さの最適入力ベクトルに置き換える。
x_optima = []

N_VALIDATION = 100
PERTURBATION_STD_RATIO = 0.01
RANDOM_SEED = 101

# CUSTOM_SIGMA を None のままにすると diag((PERTURBATION_STD_RATIO * bounds_width)^2) を使う。
# 既知の共分散行列を使う場合は shape=(cfg.n_params, cfg.n_params) の numpy array を設定する。
CUSTOM_SIGMA = None

In [ ]:
def validate_x_optima(x, n_params, lower_bounds, upper_bounds):
    x = np.asarray(x, dtype=float).reshape(-1)
    if x.size != n_params:
        raise ValueError(
            f"x_optima must have length cfg.n_params={n_params}, but got length {x.size}. "
            "Set x_optima before running validation."
        )
    below = x < lower_bounds
    above = x > upper_bounds
    if np.any(below) or np.any(above):
        bad = [
            (PARAM_NAMES[i], float(x[i]), float(lower_bounds[i]), float(upper_bounds[i]))
            for i in np.where(below | above)[0]
        ]
        raise ValueError(f"x_optima has out-of-bounds values: {bad}")
    return np.round(x, ROUND_DECIMALS)


def build_validation_sigma(custom_sigma=None):
    if custom_sigma is not None:
        sigma = np.asarray(custom_sigma, dtype=float)
    else:
        bounds_width = UPPER_BOUNDS - LOWER_BOUNDS
        perturbation_std = PERTURBATION_STD_RATIO * bounds_width
        sigma = np.diag(perturbation_std ** 2)

    if sigma.shape != (cfg.n_params, cfg.n_params):
        raise ValueError(f"Sigma must have shape {(cfg.n_params, cfg.n_params)}, got {sigma.shape}.")
    return sigma


x_center = validate_x_optima(x_optima, cfg.n_params, LOWER_BOUNDS, UPPER_BOUNDS)
VALIDATION_SIGMA = build_validation_sigma(CUSTOM_SIGMA)
rng = np.random.default_rng(RANDOM_SEED)

Z_validation = lib_gp.sample_input_perturbations(
    x=x_center,
    Sigma=VALIDATION_SIGMA,
    n_samples=N_VALIDATION,
    bounds=BOUNDS,
    rng=rng,
)
Z_validation = np.round(Z_validation, ROUND_DECIMALS)

validation_inputs_df = pd.DataFrame(Z_validation, columns=PARAM_NAMES)
validation_inputs_df.insert(0, "sample_idx", np.arange(N_VALIDATION))
display(validation_inputs_df.head())

In [ ]:
validation_rows = []
start = time.perf_counter()

try:
    for sample_idx, params in enumerate(Z_validation, start=1):
        print(f"[validation] HFSS evaluation {sample_idx}/{N_VALIDATION}")
        y, row = evaluate_hfss(PARAM_NAMES, params)
        row["sample_idx"] = sample_idx - 1
        row["center_distance_l2"] = float(np.linalg.norm(np.asarray(params, dtype=float) - x_center))
        validation_rows.append(row)

    validation_df = pd.DataFrame(validation_rows)
    y_values = validation_df["S11"].to_numpy(dtype=float)
    M = y_values.size
    mu_sample = float(np.mean(y_values))
    mu_sample_std = 0.0 if M <= 1 else float(np.sqrt(np.sum((y_values - mu_sample) ** 2) / (M - 1)))

    validation_summary = {
        "M": int(M),
        "mu_sample": mu_sample,
        "mu_sample_std": mu_sample_std,
        "std_formula": "sqrt(sum((S11_i - mu_sample)^2) / (M - 1))",
        "perturbation_std_ratio": PERTURBATION_STD_RATIO,
        "random_seed": RANDOM_SEED,
    }

    validation_csv = backbone._get_dir_run() / "validation_samples.csv"
    summary_csv = backbone._get_dir_run() / "validation_summary.csv"
    validation_df.to_csv(validation_csv, index=False)
    pd.DataFrame([validation_summary]).to_csv(summary_csv, index=False)

    # HDF5 にも保存する（文字列列を避けるため validation samples の数値列のみ）
    backbone._addNewDatasetToHDF(validation_df.select_dtypes(include=[np.number]), "output", "validation_samples")
    backbone._addNewDatasetToHDF(pd.DataFrame([validation_summary]).select_dtypes(include=[np.number]), "output", "validation_summary")

    elapsed = time.perf_counter() - start
    print("-" * 75)
    print(f"HFSS validation finished in {elapsed:.3f} seconds")
    print(f"mu_sample     = {mu_sample:.10f}")
    print(f"mu_sample_std = {mu_sample_std:.10f}")
    print(f"Saved samples : {validation_csv}")
    print(f"Saved summary : {summary_csv}")

    display(pd.DataFrame([validation_summary]))

finally:
    done_flag_path = Path(_config_temp["DONE_FLAG_FILE"])
    done_flag_path.touch()

    json_file = base_dir / Path("_config_HFSS.json")
    json_file.unlink(missing_ok=True)
    if backbone.h5f:
        backbone.h5f.close()

In [ ]:
# x_optima 周りの摂動と、S11 の期待値周りのヒストグラム
import math

import matplotlib.pyplot as plt

# HFSS 評価後に実行する想定。validation_df があれば評価済みの丸め済み入力を優先する。
if "validation_df" in globals() and not validation_df.empty:
    perturbation_values = validation_df[PARAM_NAMES].to_numpy(dtype=float) - x_center
    s11_values = validation_df["S11"].to_numpy(dtype=float)
else:
    perturbation_values = Z_validation - x_center
    s11_values = None

n_params = len(PARAM_NAMES)
n_cols = 4
n_rows = math.ceil(n_params / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.2 * n_cols, 3.0 * n_rows), constrained_layout=True)
axes = np.asarray(axes).reshape(-1)

for idx, (ax, param_name) in enumerate(zip(axes, PARAM_NAMES)):
    values = perturbation_values[:, idx]
    ax.hist(values, bins="auto", edgecolor="black", alpha=0.75)
    ax.axvline(0.0, color="tab:red", linestyle="--", linewidth=1.5, label="x_optima")
    ax.set_title(param_name)
    ax.set_xlabel(f"{param_name} - x_optima")
    ax.set_ylabel("count")
    ax.grid(alpha=0.25)
    ax.legend(loc="best")

for ax in axes[n_params:]:
    ax.axis("off")

fig.suptitle("Perturbation histograms around x_optima", fontsize=16)
plt.show()

if s11_values is None:
    print("S11 histogram is skipped because validation_df is not available yet. Run the HFSS validation cell first.")
else:
    expected_s11 = float(np.mean(s11_values))
    s11_centered = s11_values - expected_s11

    fig, ax = plt.subplots(figsize=(7.5, 4.5), constrained_layout=True)
    ax.hist(s11_centered, bins="auto", edgecolor="black", alpha=0.75)
    ax.axvline(0.0, color="tab:red", linestyle="--", linewidth=1.5, label=f"E[S11] = {expected_s11:.6g}")
    ax.set_title("S11 histogram around expected value")
    ax.set_xlabel("S11 - E[S11]")
    ax.set_ylabel("count")
    ax.grid(alpha=0.25)
    ax.legend(loc="best")
    plt.show()
